#Configuration



In [ ]:
import os
import pandas as pd
import requests
import json
import base64
import time
import glob

# DataForSEO Credentials
login = os.environ["DATAFORSEO_LOGIN"]
password = os.environ["DATAFORSEO_PASSWORD"]

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords"
if not os.path.exists(base_path):
    os.makedirs(base_path)

# Prepare Keyword CSV File

In [ ]:
# Define the input directory path
input_dir = os.path.join(base_path, "input")
if not os.path.exists(input_dir):
    os.makedirs(input_dir)
    print(f"Created directory: {input_dir}")


sample_data = {
    'location_name': ['roanoke_lynchburg', 'blacksburg', 'christiansburg','DC','NY','Charlotte'],
    'location_code': [200573, 1027041, 1027077,2840,21167,200517],
    'keywords_General_Dentist': [
        'dental clinic',
        'family dentist',
        'dentist',
        'general dentistry',
        'cosmetic dentist',
        None
    ],
    'keywords_Special_Dentist': [
        'orthodontist',
        'pediatric dentist',
        'periodontist',
        'prosthodontist',
        None,
        None
    ],
    'keywords_Surgery_Dentist': [
        'oral surgeon',
        'dental implants',
        'emergency dentist',
        None,
        None,
        None
    ]
}
sample_df = pd.DataFrame(sample_data)
csv_path = os.path.join(input_dir, "keyword_dentist.csv") # Save to input directory
sample_df.to_csv(csv_path, index=False)

print(f"Sample file saved to: {csv_path}")

from IPython.display import display
display(sample_df)

# POST Tasks to DataForSEO
## Wait 20min after running this cell

In [ ]:
def post_dataforseo_task(post_url, location_code, keyword, depth=100):
    post_payload = [{
        "location_code": location_code,
        "language_code": "en",
        "keyword": keyword,
        "depth": depth
    }]

    cred_string = f"{login}:{password}"
    cred_base64 = base64.b64encode(cred_string.encode("utf-8")).decode("utf-8")
    headers = {
        'Authorization': f'Basic {cred_base64}',
        'Content-Type': 'application/json'
    }

    print(f"  POST task: Location Code={location_code}, Keyword='{keyword}'")

    try:
        response = requests.post(post_url, headers=headers, json=post_payload)
        response.raise_for_status()
        result = response.json()

        if result and result.get("tasks") and result["tasks"][0].get("id"):
            task_id = result["tasks"][0]["id"]
            print(f"  Task POST successfully for keyword: '{keyword}'. Task ID: {task_id}")
            return task_id
        else:
            print(f"  Task POST successful, but no Task ID found in response. Keyword: '{keyword}'. Response: {result}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"  Error POST task for keyword: '{keyword}': {e}")
        return None
    except json.JSONDecodeError:
        print(f"  Task POST successful, but could not parse JSON response. Keyword: '{keyword}'. Response text: {response.text}")
        return None




In [ ]:
print("--- Starting Task POST ---")
input_dir = os.path.join(base_path, "input")
temp_dir = os.path.join(base_path, "temp")
output_dir = os.path.join(base_path, "output")

if not os.path.exists(input_dir):
    os.makedirs(input_dir)
    print(f"Created directory: {input_dir}")
if not os.path.exists(temp_dir):
    os.makedirs(temp_dir)
    print(f"Created directory: {temp_dir}")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")


csv_path = os.path.join(input_dir, "keyword_dentist.csv")
try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    print(f"Error: '{csv_path}' not found. Please run the previous cell first.")
    tasks_to_submit_df = pd.DataFrame()
else:
    tasks_to_submit_df = df.copy()

task_list_csv_path = os.path.join(temp_dir, "task_list.csv")

# Load already posted tasks
existing_tasks = set()
try:
    existing_tasks_df = pd.read_csv(task_list_csv_path)
    # Create a set of (location_name, keyword, api_type) tuples
    for index, row in existing_tasks_df.iterrows():
        existing_tasks.add((row['location_name'], row['keyword'], row['api_type']))
    print(f"Loaded {len(existing_tasks)} existing tasks to skip.")
except FileNotFoundError:
    print("No existing task file found. Will post all tasks.")


write_header = not os.path.exists(task_list_csv_path)

api_endpoints = {
    "local_finder": "https://api.dataforseo.com/v3/serp/google/local_finder/task_post",
    "maps": "https://api.dataforseo.com/v3/serp/google/maps/task_post"
}

if not tasks_to_submit_df.empty:
    keyword_columns = [col for col in tasks_to_submit_df.columns if col.startswith('keyword')]

    if not keyword_columns:
        print("Error: No keyword columns found.")
    else:
        for keyword_col in keyword_columns:
            print(f"Processing keyword column: '{keyword_col}'")
            keywords_in_column = tasks_to_submit_df[keyword_col].dropna().tolist()

            if not keywords_in_column:
                print(f"  Keyword column '{keyword_col}' has no keywords. Skipping.")
                continue

            from itertools import combinations
            keyword_combinations = []
            # for i in range(1, min(len(keywords_in_column), 3) + 1):
            for i in range(1, len(keywords_in_column) + 1):
                 keyword_combinations.extend(list(combinations(keywords_in_column, i)))

            if not keyword_combinations:
                 print(f"  Could not generate combinations for keyword column '{keyword_col}'. Skipping.")
                 continue

            print(f"  Keyword combinations for column '{keyword_col}': {keyword_combinations}")

            for location_index, location_row in tasks_to_submit_df.iterrows():
                location_name = location_row['location_name']
                location_code_val = location_row['location_code']

                if pd.isna(location_code_val):
                    print(f"  Skipping task POST for location '{location_name}' due to missing location_code.")
                    continue

                location_code = int(location_code_val)
                print(f"  Processing tasks for location '{location_name}' ({location_code}):")

                for combo in keyword_combinations:
                    combined_keyword = "+".join(combo)

                    for api_name, api_url in api_endpoints.items():
                        # Skip posted task
                        if (location_name, combined_keyword, api_name) in existing_tasks:
                            print(f"    Skipping already posted task: Keyword='{combined_keyword}', API='{api_name}'")
                            continue

                        print(f"    Combined keyword: '{combined_keyword}'")
                        raw_output_dir = os.path.join(temp_dir, api_name)
                        if not os.path.exists(raw_output_dir):
                            os.makedirs(raw_output_dir)
                            print(f"Created directory: {raw_output_dir}")

                        task_id = post_dataforseo_task(api_url, location_code, combined_keyword)
                        if task_id:
                            safe_combined_keyword = combined_keyword.replace(' ', '_').replace('/', '_').replace('\\\\', '_')
                            raw_json_filename = f"{location_name}_{safe_combined_keyword}_{api_name}.json"
                            raw_json_path = os.path.join(raw_output_dir, raw_json_filename)

                            current_task_df = pd.DataFrame([{
                                "task_id": task_id,
                                "api_type": api_name,
                                "location_name": location_name,
                                "keyword": combined_keyword,
                                "raw_json_path": raw_json_path
                            }])

                            current_task_df.to_csv(task_list_csv_path, mode='a', header=write_header, index=False)
                            write_header = False
                            print(f"  Task info appended to: '{task_list_csv_path}'")

                        time.sleep(1)


    print(f"All new tasks posted and incrementally saved to '{task_list_csv_path}').")
    print("!!! IMPORTANT: Please wait 20 minutes before running the next cell to allow results to become available. !!!")
else:
    print("Keyword database file not found or empty.")

In [ ]:
#仅跑一次，因为api关于dc的location code更新了，仅跑dc
import pandas as pd
import os
import requests
import json
import base64
import time
import glob
from itertools import combinations

print("--- Starting Task POST ---")
input_dir = os.path.join(base_path, "input")
temp_dir = os.path.join(base_path, "temp")
output_dir = os.path.join(base_path, "output")

if not os.path.exists(input_dir):
    os.makedirs(input_dir)
    print(f"Created directory: {input_dir}")
if not os.path.exists(temp_dir):
    os.makedirs(temp_dir)
    print(f"Created directory: {temp_dir}")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")


csv_path = os.path.join(input_dir, "keyword_dentist.csv")
try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    print(f"Error: '{csv_path}' not found. Please run the previous cell first.")
    tasks_to_submit_df = pd.DataFrame()
else:
    tasks_to_submit_df = df.copy()

task_list_csv_path = os.path.join(temp_dir, "task_list.csv")

# Load already posted tasks
existing_tasks = set()
try:
    existing_tasks_df = pd.read_csv(task_list_csv_path)
    # Create a set of (location_name, keyword, api_type) tuples
    for index, row in existing_tasks_df.iterrows():
        existing_tasks.add((row['location_name'], row['keyword'], row['api_type']))
    print(f"Loaded {len(existing_tasks)} existing tasks to skip.")
except FileNotFoundError:
    print("No existing task file found. Will post all tasks.")


write_header = not os.path.exists(task_list_csv_path)

api_endpoints = {
    "local_finder": "https://api.dataforseo.com/v3/serp/google/local_finder/task_post",
    "maps": "https://api.dataforseo.com/v3/serp/google/maps/task_post"
}


LOCATION_TO_RERUN = 'DC'

if not tasks_to_submit_df.empty:
    keyword_columns = [col for col in tasks_to_submit_df.columns if col.startswith('keyword')]

    if not keyword_columns:
        print("Error: No keyword columns found.")
    else:
        for keyword_col in keyword_columns:
            print(f"Processing keyword column: '{keyword_col}'")
            keywords_in_column = tasks_to_submit_df[keyword_col].dropna().tolist()

            if not keywords_in_column:
                print(f"  Keyword column '{keyword_col}' has no keywords. Skipping.")
                continue

            from itertools import combinations
            keyword_combinations = []
            for i in range(1, len(keywords_in_column) + 1):
                keyword_combinations.extend(list(combinations(keywords_in_column, i)))

            if not keyword_combinations:
                print(f"  Could not generate combinations for keyword column '{keyword_col}'. Skipping.")
                continue

            print(f"  Keyword combinations for column '{keyword_col}': {keyword_combinations}")

            for location_index, location_row in tasks_to_submit_df.iterrows():
                location_name = location_row['location_name']
                location_code_val = location_row['location_code']

                if pd.isna(location_code_val):
                    print(f"  Skipping task POST for location '{location_name}' due to missing location_code.")
                    continue

                location_code = int(location_code_val)
                print(f"  Processing tasks for location '{location_name}' ({location_code}):")

                for combo in keyword_combinations:
                    combined_keyword = "+".join(combo)

                    for api_name, api_url in api_endpoints.items():


                        # Check if the task is already in the list
                        if (location_name, combined_keyword, api_name) in existing_tasks:

                            # skip if it's not the location we want to rerun
                            if location_name != LOCATION_TO_RERUN:
                                print(f"    Skipping already posted task: Keyword='{combined_keyword}', API='{api_name}'")
                                continue
                            else:

                                print(f"    RE-POSTING task for '{location_name}': Keyword='{combined_keyword}'")

                        print(f"    Posting new/overwriting task with keyword: '{combined_keyword}'")
                        raw_output_dir = os.path.join(temp_dir, api_name)
                        if not os.path.exists(raw_output_dir):
                            os.makedirs(raw_output_dir)
                            print(f"Created directory: {raw_output_dir}")

                        task_id = post_dataforseo_task(api_url, location_code, combined_keyword)
                        if task_id:
                            safe_combined_keyword = combined_keyword.replace(' ', '_').replace('/', '_').replace('\\\\', '_')
                            raw_json_filename = f"{location_name}_{safe_combined_keyword}_{api_name}.json"
                            raw_json_path = os.path.join(raw_output_dir, raw_json_filename)

                            current_task_df = pd.DataFrame([{
                                "task_id": task_id,
                                "api_type": api_name,
                                "location_name": location_name,
                                "keyword": combined_keyword,
                                "raw_json_path": raw_json_path
                            }])

                            current_task_df.to_csv(task_list_csv_path, mode='a', header=write_header, index=False)
                            write_header = False
                            print(f"  Task info appended to: '{task_list_csv_path}'")

                        time.sleep(1)


    print(f"All new tasks posted and incrementally saved to '{task_list_csv_path}').")
    print("!!! IMPORTANT: Please wait 20 minutes before running the next cell to allow results to become available. !!!")
else:
    print("Keyword database file not found or empty.")

#Get Task Results
## Wait 20 mins after running previous cell

In [ ]:
def get_dataforseo_results(task_id, api_type, raw_json_path):
    if api_type == "local_finder":
        get_url_template = "https://api.dataforseo.com/v3/serp/google/local_finder/task_get/advanced/{}"
    elif api_type == "maps":
        get_url_template = "https://api.dataforseo.com/v3/serp/google/maps/task_get/advanced/{}"
    else:
        print(f"  Unknown API type: {api_type}")
        return

    get_url = get_url_template.format(task_id)

    cred_string = f"{login}:{password}"
    cred_base64 = base64.b64encode(cred_string.encode("utf-8")).decode("utf-8")
    headers = {'Authorization': f'Basic {cred_base64}'}

    try:
        response = requests.get(get_url, headers=headers)
        response.raise_for_status()
        result_data = response.json()

        raw_data_dir = os.path.dirname(raw_json_path)
        if not os.path.exists(raw_data_dir):
            os.makedirs(raw_data_dir)
            print(f"Created directory: {raw_data_dir}")

        with open(raw_json_path, "w", encoding="utf-8") as f:
            json.dump(result_data, f, indent=4, ensure_ascii=False)
        print(f"  Saved JSON results for Task ID {task_id} to {os.path.basename(raw_json_path)}") # Updated message
    except requests.exceptions.RequestException as e:
        print(f"  Error getting results for Task ID {task_id}: {e}")

In [ ]:
print("--- Get Task Results ---")
temp_dir = os.path.join(base_path, "temp")
task_list_path = os.path.join(temp_dir, "task_list.csv")
tasks_to_get = []
try:
    task_list_df = pd.read_csv(task_list_path)
    tasks_to_get = task_list_df.to_dict('records')
except FileNotFoundError:
    print(f"Error: '{task_list_path}' not found. Please run the task submission cell first.")

if tasks_to_get:
    print(f"Found {len(tasks_to_get)} tasks to get results for.")
    for task in tasks_to_get:

        # Skip result file already exists
        if os.path.exists(task['raw_json_path']):
            print(f"  Skipping: Result file already exists -> '{os.path.basename(task['raw_json_path'])}'")
            continue

        print(f" Getting results for: Location='{task['location_name']}', Keyword='{task['keyword']}'")
        get_dataforseo_results(task['task_id'], task['api_type'], task['raw_json_path'])
        time.sleep(1)

    print("\nAll result retrieval attempts completed.")

In [ ]:
#仅一次，dc
print("--- Get Task Results ---")
temp_dir = os.path.join(base_path, "temp")
task_list_path = os.path.join(temp_dir, "task_list.csv")
tasks_to_get = []
try:
    task_list_df = pd.read_csv(task_list_path)
    tasks_to_get = task_list_df.to_dict('records')
except FileNotFoundError:
    print(f"Error: '{task_list_path}' not found. Please run the task submission cell first.")


LOCATION_TO_RERUN = 'DC'

if tasks_to_get:
    print(f"Found {len(tasks_to_get)} tasks to get results for.")
    for task in tasks_to_get:

        if os.path.exists(task['raw_json_path']):

            if task['location_name'] != LOCATION_TO_RERUN:
                print(f"   Skipping: Result file already exists -> '{os.path.basename(task['raw_json_path'])}'")
                continue
            else:
                print(f"   RE-GETTING (overwriting) results for '{LOCATION_TO_RERUN}': Keyword='{task['keyword']}'")

        print(f" Getting results for: Location='{task['location_name']}', Keyword='{task['keyword']}'")
        get_dataforseo_results(task['task_id'], task['api_type'], task['raw_json_path'])
        time.sleep(1)

    print("\nAll result retrieval attempts completed.")

# Process Data

In [ ]:
KEYWORD_CATEGORIES = {
    'keywords_General_Dentist': [
        'dental clinic', 'family dentist', 'dentist',
        'general dentistry', 'cosmetic dentist',
    ],
    'keywords_Special_Dentist': [
        'orthodontist', 'pediatric dentist', 'periodontist', 'prosthodontist',
    ],
    'keywords_Surgery_Dentist': [
        'oral surgeon', 'dental implants', 'emergency dentist',
    ]
}

keyword_to_category_map = {
    keyword: category
    for category, keywords in KEYWORD_CATEGORIES.items()
    for keyword in keywords
}


def get_category_from_keywords(keyword_list):
    for keyword in keyword_list:
        if keyword in keyword_to_category_map:
            return keyword_to_category_map[keyword]
    return 'Unknown'

In [ ]:
LOCATION_PREFIXES = [
'roanoke_lynchburg_', 'blacksburg_', 'christiansburg_','DC_','NY_','Charlotte_'
]

def parse_local_finder_results(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Could not read or parse {os.path.basename(file_path)}: {e}")
        return []

    file_name = os.path.basename(file_path)
    try:
        keywords_str = file_name

        # remove LOCATION_PREFIXES
        for prefix in LOCATION_PREFIXES:
            if keywords_str.startswith(prefix):
                keywords_str = keywords_str.replace(prefix, '', 1)
                break

        # Clean the suffixes
        keywords_str = keywords_str.replace('_local_finder.json', '').replace('_maps.json', '')

        # Extract keywords
        keywords_list = [kw.replace('_', ' ') for kw in keywords_str.split('+')]
        formatted_keywords = ', '.join(keywords_list)
        category = get_category_from_keywords(keywords_list)

    except Exception as e:
        print(f"Error processing filename {file_name}: {e}")
        formatted_keywords = 'N/A'
        category = 'Unknown'

    extracted_data = []
    if not (data and data.get("tasks") and data["tasks"][0].get("result")):
        return []

    for result in data["tasks"][0]["result"]:
        if not result or not result.get("items"):
            continue
        for item in result["items"]:
            rating = item.get("rating", {})
            if not isinstance(rating, dict): rating = {}

            extracted_data.append({
                "title": item.get("title"),
                "description": item.get("description"),
                "rating_value": rating.get("value"),
                "votes_count": rating.get("votes_count"),
                "type": item.get("type"),
                "keywords": formatted_keywords,
                "category": category
            })
    return extracted_data

def parse_maps_results(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Could not read or parse {os.path.basename(file_path)}: {e}")
        return []


    file_name = os.path.basename(file_path)
    try:
        keywords_str = file_name

        for prefix in LOCATION_PREFIXES:
            if keywords_str.startswith(prefix):
                keywords_str = keywords_str.replace(prefix, '', 1)
                break

        keywords_str = keywords_str.replace('_local_finder.json', '').replace('_maps.json', '')

        keywords_list = [kw.replace('_', ' ') for kw in keywords_str.split('+')]
        formatted_keywords = ', '.join(keywords_list)
        category = get_category_from_keywords(keywords_list)

    except Exception as e:
        print(f"Error processing filename {file_name}: {e}")
        formatted_keywords = 'N/A'
        category = 'Unknown'

    extracted_data = []
    if not (data and data.get("tasks") and data["tasks"][0].get("result")):
        return []

    for result in data["tasks"][0]["result"]:
        if not result or not result.get("items"):
            continue
        for item in result["items"]:
            rating = item.get("rating", {})
            if not isinstance(rating, dict): rating = {}
            rating_distribution = item.get("rating_distribution", {})
            if not isinstance(rating_distribution, dict): rating_distribution = {}
            address_info = item.get("address_info", {})
            if not isinstance(address_info, dict): address_info = {}

            extracted_data.append({
                "title": item.get("title"),
                "address": item.get("address"),
                "latitude": item.get("latitude"),
                "longitude": item.get("longitude"),
                "zip": address_info.get("zip"),
                "rating_value": rating.get("value"),
                "votes_count": rating.get("votes_count"),
                "rating_1_star": rating_distribution.get("1", 0),
                "rating_2_star": rating_distribution.get("2", 0),
                "rating_3_star": rating_distribution.get("3", 0),
                "rating_4_star": rating_distribution.get("4", 0),
                "rating_5_star": rating_distribution.get("5", 0),
                "type": item.get("type"),
                "keywords": formatted_keywords,
                "category": category
            })
    return extracted_data


In [ ]:
# print("--- Data Processing ---")

# temp_dir = os.path.join(base_path, "temp")
# output_dir = os.path.join(base_path, "output")

# local_finder_raw_json_dir = os.path.join(temp_dir, "local_finder")
# maps_raw_json_dir = os.path.join(temp_dir, "maps")

# local_finder_output_raw_dir = os.path.join(output_dir, "local_finder", "raw") # before deduplication
# local_finder_output_processed_dir = os.path.join(output_dir, "local_finder", "processed") # processed data
# maps_output_raw_dir = os.path.join(output_dir, "maps", "raw") # before deduplication
# maps_output_processed_dir = os.path.join(output_dir, "maps", "processed") # processed data

# for d in [local_finder_output_raw_dir, local_finder_output_processed_dir, maps_output_raw_dir, maps_output_processed_dir]:
#     if not os.path.exists(d):
#         os.makedirs(d)
#         print(f"Created directory: {d}")


# local_finder_json_files = glob.glob(os.path.join(local_finder_raw_json_dir, "*.json"))
# maps_json_files = glob.glob(os.path.join(maps_raw_json_dir, "*.json"))

# local_finder_data = []
# maps_data = []

# if not local_finder_json_files and not maps_json_files:
#     print("No JSON result files found for processing in temp/local_finder or temp/maps.")
# else:
#     print(f"Found {len(local_finder_json_files)} Local Finder files and {len(maps_json_files)} Maps files to process.") # Updated message

#     # Process Local Finder files
#     for file in local_finder_json_files:
#         print(f"Processing Local Finder file: {os.path.basename(file)}")
#         local_finder_data.extend(parse_local_finder_results(file))

#     # Process Maps files
#     for file in maps_json_files:
#         print(f"Processing Maps file: {os.path.basename(file)}")
#         maps_data.extend(parse_maps_results(file))


#     # Process Local Finder data
#     if local_finder_data:
#         local_finder_df = pd.DataFrame(local_finder_data)
#         print(f"\nLocal Finder locations before deduplication: {len(local_finder_df)}")

#         local_finder_raw_csv_path = os.path.join(local_finder_output_raw_dir, "local_finder_raw_summary.csv")
#         local_finder_df.to_csv(local_finder_raw_csv_path, index=False, encoding='utf-8-sig')
#         print(f"Successfully saved Local Finder data BEFORE deduplication to: '{local_finder_raw_csv_path}'")


#         # Deduplicate based on title
#         local_finder_deduplicated_df = local_finder_df.drop_duplicates(subset=['title'], keep='first')
#         print(f"Local Finder locations after deduplication: {len(local_finder_deduplicated_df)}")

#         local_finder_sorted_df = local_finder_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

#         local_finder_processed_csv_path = os.path.join(local_finder_output_processed_dir, "local_finder_processed_summary.csv")
#         local_finder_sorted_df.to_csv(local_finder_processed_csv_path, index=False, encoding='utf-8-sig')
#         print(f"\nSaved deduplicated Local Finder data to: '{local_finder_processed_csv_path}'")

#         print("\nFinal Local Finder data preview:")
#         display(local_finder_sorted_df.head(10))
#     else:
#         print("\nNo data extracted from Local Finder files.")

#     # Process Maps data
#     if maps_data:
#         maps_df = pd.DataFrame(maps_data)
#         print(f"\nMaps locations before deduplication: {len(maps_df)}")

#         maps_raw_csv_path = os.path.join(maps_output_raw_dir, "maps_raw_summary.csv")
#         maps_df.to_csv(maps_raw_csv_path, index=False, encoding='utf-8-sig')
#         print(f"Successfully saved Maps data BEFORE deduplication to: '{maps_raw_csv_path}'")

#          # Deduplicate based on title and address
#         maps_deduplicated_df = maps_df.drop_duplicates(subset=['title', 'address'], keep='first')
#         print(f"Maps locations after deduplication: {len(maps_deduplicated_df)}")

#         maps_sorted_df = maps_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')
#         maps_processed_csv_path = os.path.join(maps_output_processed_dir, "maps_processed_summary.csv")
#         maps_sorted_df.to_csv(maps_processed_csv_path, index=False, encoding='utf-8-sig')
#         print(f"\nSuccessfully saved deduplicated Maps data to: '{maps_processed_csv_path}'")


#         print("\nFinal Maps data preview:")
#         display(maps_sorted_df.head(10))
#     else:
#          print("\nNo data extracted from Maps files.")


#     print("\nAll result processing attempts completed.")

In [ ]:
print("--- Data Processing  ---")

temp_dir = os.path.join(base_path, "temp")
output_dir = os.path.join(base_path, "output")

local_finder_raw_json_dir = os.path.join(temp_dir, "local_finder")
maps_raw_json_dir = os.path.join(temp_dir, "maps")

local_finder_output_raw_dir = os.path.join(output_dir, "local_finder", "raw")
local_finder_output_processed_dir = os.path.join(output_dir, "local_finder", "processed")
maps_output_raw_dir = os.path.join(output_dir, "maps", "raw")
maps_output_processed_dir = os.path.join(output_dir, "maps", "processed")

for d in [local_finder_output_raw_dir, local_finder_output_processed_dir, maps_output_raw_dir, maps_output_processed_dir]:
    os.makedirs(d, exist_ok=True)

# log file
local_finder_log_path = os.path.join(local_finder_raw_json_dir, "_processed_files.log")
maps_log_path = os.path.join(maps_raw_json_dir, "_processed_files.log")

def load_processed_log(log_path):
    # Load files already processed
    try:
        with open(log_path, 'r') as f:
            return set(line.strip() for line in f)
    except FileNotFoundError:
        return set()

def update_processed_log(log_path, new_files):
    # Append newly processed filenames to the log
    with open(log_path, 'a') as f:
        for file_name in new_files:
            f.write(f"{file_name}\n")

def load_existing_data(csv_path):
    # Load the previously processed CSV data
    try:
        return pd.read_csv(csv_path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

# Load Logs
processed_lf_set = load_processed_log(local_finder_log_path)
processed_maps_set = load_processed_log(maps_log_path)
print(f"Loaded {len(processed_lf_set)} processed Local Finder file records from log.")
print(f"Loaded {len(processed_maps_set)} processed Maps file records from log.")

local_finder_json_files = glob.glob(os.path.join(local_finder_raw_json_dir, "*.json"))
maps_json_files = glob.glob(os.path.join(maps_raw_json_dir, "*.json"))

if not local_finder_json_files and not maps_json_files:
    print("No JSON result files found for processing in temp/local_finder or temp/maps.")
else:
    print(f"Found {len(local_finder_json_files)} total Local Finder files and {len(maps_json_files)} total Maps files.")

    # local finder
    new_local_finder_data = []
    processed_this_run_lf = []

    for file_path in local_finder_json_files:
        file_name = os.path.basename(file_path)
        if file_name in processed_lf_set:
            continue  # Skip if in log

        print(f"Processing new Local Finder file: {file_name}")
        parsed_data = parse_local_finder_results(file_path)
        if parsed_data:
            new_local_finder_data.extend(parsed_data)
            processed_this_run_lf.append(file_name)

    # map
    new_maps_data = []
    processed_this_run_maps = []

    for file_path in maps_json_files:
        file_name = os.path.basename(file_path)
        if file_name in processed_maps_set:
            continue  # Skip if in log

        print(f"Processing new Maps file: {file_name}")
        parsed_data = parse_maps_results(file_path)
        if parsed_data:
            new_maps_data.extend(parsed_data)
            processed_this_run_maps.append(file_name)

    # Consolidate Local Finder data
    local_finder_processed_csv_path = os.path.join(local_finder_output_processed_dir, "local_finder_processed_summary.csv")

    if new_local_finder_data:
        df_new_lf = pd.DataFrame(new_local_finder_data)
        df_old_lf = load_existing_data(local_finder_processed_csv_path)
        df_all_lf = pd.concat([df_old_lf, df_new_lf], ignore_index=True)
        print(f"\nLoaded {len(df_old_lf)} old Local Finder records. {len(df_new_lf)} new records added.")

        local_finder_raw_csv_path = os.path.join(local_finder_output_raw_dir, "local_finder_raw_summary.csv")
        df_all_lf.to_csv(local_finder_raw_csv_path, index=False, encoding='utf-8-sig')
        print(f"Local Finder locations before deduplication: {len(df_all_lf)}")

        local_finder_deduplicated_df = df_all_lf.drop_duplicates(subset=['title'], keep='first')
        print(f"Local Finder locations after deduplication: {len(local_finder_deduplicated_df)}")
        local_finder_sorted_df = local_finder_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

        local_finder_sorted_df.to_csv(local_finder_processed_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSaved deduplicated Local Finder data to: '{local_finder_processed_csv_path}'")
        display(local_finder_sorted_df.head(10))

        update_processed_log(local_finder_log_path, processed_this_run_lf)
        print(f"Updated Local Finder log with {len(processed_this_run_lf)} new files.")
    else:
        print("\nNo new Local Finder files to process.")

    # Consolidate Maps data
    maps_processed_csv_path = os.path.join(maps_output_processed_dir, "maps_processed_summary.csv")

    if new_maps_data:
        df_new_maps = pd.DataFrame(new_maps_data)
        df_old_maps = load_existing_data(maps_processed_csv_path)
        df_all_maps = pd.concat([df_old_maps, df_new_maps], ignore_index=True)
        print(f"\nLoaded {len(df_old_maps)} old Maps records. {len(df_new_maps)} new records added.")

        maps_raw_csv_path = os.path.join(maps_output_raw_dir, "maps_raw_summary.csv")
        df_all_maps.to_csv(maps_raw_csv_path, index=False, encoding='utf-8-sig')
        print(f"Maps locations before deduplication: {len(df_all_maps)}")

        maps_deduplicated_df = df_all_maps.drop_duplicates(subset=['title', 'address'], keep='first')
        print(f"Maps locations after deduplication: {len(maps_deduplicated_df)}")
        maps_sorted_df = maps_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

        maps_sorted_df.to_csv(maps_processed_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSuccessfully saved deduplicated Maps data to: '{maps_processed_csv_path}'")
        display(maps_sorted_df.head(10))

        update_processed_log(maps_log_path, processed_this_run_maps)
        print(f"Updated Maps log with {len(processed_this_run_maps)} new files.")
    else:
        print("\nNo new Maps files to process.")

    print("\nAll result processing attempts completed.")

# Merge and Deduplicate Processed Results

In [ ]:
print("--- Merging Processed Results ---")

output_dir = os.path.join(base_path, "output")
local_finder_processed_path = os.path.join(output_dir, "local_finder", "processed", "local_finder_processed_summary.csv")
maps_processed_path = os.path.join(output_dir, "maps", "processed", "maps_processed_summary.csv")

local_finder_processed_df = pd.DataFrame()
maps_processed_df = pd.DataFrame()

try:
    local_finder_processed_df = pd.read_csv(local_finder_processed_path)
    print(f"Loaded Local Finder processed data from: '{local_finder_processed_path}'")
except FileNotFoundError:
    print(f"Error: '{local_finder_processed_path}' not found.")

try:
    maps_processed_df = pd.read_csv(maps_processed_path)
    print(f"Loaded Maps processed data from: '{maps_processed_path}'")
except FileNotFoundError:
    print(f"Error: '{maps_processed_path}' not found.")

# Check if both df were loaded successfully
if not local_finder_processed_df.empty and not maps_processed_df.empty:


    # Merge the two df based on 'title'
    merged_df = pd.merge(local_finder_processed_df, maps_processed_df, on='title', how='outer', suffixes=('_local_finder', '_maps'))

    print(f"Merged df shape: {merged_df.shape}")

    # Create indicator columns based is in which or both api
    merged_df['isFinder'] = merged_df['rating_value_local_finder'].apply(lambda x: 0 if pd.isna(x) else 1)
    merged_df['isMap'] = merged_df['rating_value_maps'].apply(lambda x: 0 if pd.isna(x) else 1)
    merged_df['isBoth'] = merged_df.apply(lambda row: 1 if row['isFinder'] == 1 and row['isMap'] == 1 else 0, axis=1)

    merged_df['keywords'] = merged_df['keywords_local_finder'].fillna(merged_df['keywords_maps'])
    merged_df['category'] = merged_df['category_local_finder'].fillna(merged_df['category_maps'])

    merged_df['votes_count'] = merged_df['votes_count_local_finder'].fillna(merged_df['votes_count_maps'])
    merged_df['rating_value'] = merged_df['rating_value_local_finder'].fillna(merged_df['rating_value_maps'])

    for i in range(1, 6):
        star_col = f'rating_{i}_star'
        if star_col in merged_df.columns:
            merged_df[star_col] = merged_df[star_col].fillna(0)
        else:
            merged_df[star_col] = 0

    final_df = merged_df[[
        'title', 'keywords', 'category',
        'votes_count', 'rating_value', 'address', 'latitude', 'longitude','zip',
        'rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star',
        'isFinder', 'isMap', 'isBoth'
    ]].copy()


    print(f"\nCombined df shape before deduplication: {final_df.shape}")

    # Deduplicate the combined df based on 'title'
    final_deduplicated_df = final_df.drop_duplicates(subset=['title'], keep='first')

    print(f"Combined df shape after deduplication: {final_deduplicated_df.shape}")

    # Sort the final df by rating_value and votes_count
    final_sorted_df = final_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

    print("\nFinal Data Preview:")
    display(final_sorted_df.head(10))


    # save
    final_output_dir = os.path.join(output_dir, "merged_deduplicated")
    if not os.path.exists(final_output_dir):
        os.makedirs(final_output_dir)
        print(f"Created directory: {final_output_dir}")

    final_csv_path = os.path.join(final_output_dir, "merged_deduplicated_summary.csv")
    final_sorted_df.to_csv(final_csv_path, index=False, encoding='utf-8-sig') # Save sorted dataframe
    print(f"\nSaved final data to: '{final_csv_path}'")


elif local_finder_processed_df.empty:
    print("\nCannot merge. Local Finder processed data not found.")
elif maps_processed_df.empty:
     print("\nCannot merge. Maps processed data not found.")

# Get isFinder==1 but isMap==0 as new keyword and POST task

In [ ]:
print("--- Posting Supplementary Tasks ---")

output_dir = os.path.join(base_path, "output")
input_dir = os.path.join(base_path, "input")
temp_dir = os.path.join(base_path, "temp")

supplement_task_list_csv_path = os.path.join(temp_dir, "supplement_task_list.csv")


existing_tasks = set()
try:
    existing_tasks_df = pd.read_csv(supplement_task_list_csv_path)
    for index, row in existing_tasks_df.iterrows():
        existing_tasks.add((row['location_name'], row['keyword']))
    print(f"Loaded {len(existing_tasks)} existing supplementary tasks to skip.")
except FileNotFoundError:
    print("No existing supplementary task file found. Will post all tasks.")


merged_deduplicated_path = os.path.join(output_dir, "merged_deduplicated", "merged_deduplicated_summary.csv")
try:
    merged_df = pd.read_csv(merged_deduplicated_path)
    print(f"Loaded merged data from: '{merged_deduplicated_path}'")
except FileNotFoundError:
    print(f"Error: '{merged_deduplicated_path}' not found. Please run the previous cells first.")
    merged_df = pd.DataFrame()

if not merged_df.empty:
    supplement_keywords_df = merged_df[(merged_df['isFinder'] == 1) & (merged_df['isMap'] == 0)]
    supplement_keywords = supplement_keywords_df['title'].dropna().unique().tolist()

    print(f"Found {len(supplement_keywords)} keywords to supplement from Maps API.")

    if supplement_keywords:
        keyword_db_path = os.path.join(input_dir, "keyword_dentist.csv")
        try:
            locations_df = pd.read_csv(keyword_db_path)
        except FileNotFoundError:
            print(f"Error: Original keyword database '{keyword_db_path}' not found.")
            locations_df = pd.DataFrame()

        if not locations_df.empty:
            write_header = not os.path.exists(supplement_task_list_csv_path)
            maps_api_url = "https://api.dataforseo.com/v3/serp/google/maps/task_post"

            supplement_maps_output_dir = os.path.join(temp_dir, "maps_supplement")
            if not os.path.exists(supplement_maps_output_dir):
                os.makedirs(supplement_maps_output_dir)
                print(f"Created directory: {supplement_maps_output_dir}")

            for _, location_row in locations_df.iterrows():
                if pd.isna(location_row['location_code']):
                    print(f"Skip NAN location code")
                    continue

                location_name = location_row['location_name']
                location_code = int(location_row['location_code'])

                print(f"\nProcessing supplementary tasks for location '{location_name}' ({location_code}):")
                for keyword in supplement_keywords:
                    # check if existing task, if so, skip
                    if (location_name, keyword) in existing_tasks:
                        print(f"  Skipping already posted task for keyword: '{keyword}'")
                        continue

                    task_id = post_dataforseo_task(maps_api_url, location_code, keyword)

                    if task_id:
                        safe_keyword = keyword.replace(' ', '_').replace('/', '_').replace('\\\\', '_')
                        raw_json_filename = f"{location_name}_{safe_keyword}_maps.json"
                        raw_json_path = os.path.join(supplement_maps_output_dir, raw_json_filename)

                        current_task_df = pd.DataFrame([{
                            "task_id": task_id,
                            "api_type": "maps",
                            "location_name": location_name,
                            "keyword": keyword,
                            "raw_json_path": raw_json_path
                        }])

                        current_task_df.to_csv(supplement_task_list_csv_path, mode='a', header=write_header, index=False)

                        write_header = False

                        print(f"  Task info for '{keyword}' appended to: '{supplement_task_list_csv_path}'")

                    time.sleep(1)

            print(f"All supplementary tasks posted and incrementally saved.")
            print("!!! IMPORTANT: Please wait 20 minutes before running the next cell. !!!")
else:
    print("\nMerged data is empty. Cannot proceed.")



# Get supplement result

In [ ]:
print("--- Get & Process Supplementary Task Results ---")

temp_dir = os.path.join(base_path, "temp")
output_dir = os.path.join(base_path, "output")
supplement_task_list_path = os.path.join(temp_dir, "supplement_task_list.csv")

# Get Task Results
tasks_to_get = []
try:
    supplement_task_df = pd.read_csv(supplement_task_list_path)
    tasks_to_get = supplement_task_df.to_dict('records')
    print(f"Found {len(tasks_to_get)} supplementary tasks to get results for.")

    for task in tasks_to_get:
        # Skip existing files
        if os.path.exists(task['raw_json_path']):
            print(f"  Skipping: Result file already exists -> '{os.path.basename(task['raw_json_path'])}'")
            continue

        print(f" Getting results for: Location='{task['location_name']}', Keyword='{task['keyword']}'")
        get_dataforseo_results(task['task_id'], task['api_type'], task['raw_json_path'])
        time.sleep(1)

    print("\nAll supplementary result retrieval attempts completed.")

except FileNotFoundError:
    print(f"Error: '{supplement_task_list_path}' not found. No supplementary tasks to process.")





In [ ]:
# # Process Data
# if tasks_to_get:
#     maps_supplement_raw_json_dir = os.path.join(temp_dir, "maps_supplement")
#     supplement_json_files = glob.glob(os.path.join(maps_supplement_raw_json_dir, "*.json"))
#     supplement_data = []

#     print(f"\nFound {len(supplement_json_files)} supplementary JSON files to process.")

#     for file in supplement_json_files:
#         print(f"Processing supplementary file: {os.path.basename(file)}")
#         supplement_data.extend(parse_maps_results(file))

#     if supplement_data:
#         supplement_df = pd.DataFrame(supplement_data)
#         print(f"\nSupplementary locations before deduplication: {len(supplement_df)}")

#         supplement_deduplicated_df = supplement_df.drop_duplicates(subset=['title', 'address'], keep='first')
#         print(f"Supplementary locations after deduplication: {len(supplement_deduplicated_df)}")

#         supplement_output_dir = os.path.join(output_dir, "supplement")
#         if not os.path.exists(supplement_output_dir):
#             os.makedirs(supplement_output_dir)

#         supplement_csv_path = os.path.join(supplement_output_dir, "supplement_processed_summary.csv")
#         supplement_deduplicated_df.to_csv(supplement_csv_path, index=False, encoding='utf-8-sig')
#         print(f"\nSaved processed supplementary data to: '{supplement_csv_path}'")
#         display(supplement_deduplicated_df.head())
#     else:
#         print("\nNo data extracted from supplementary Maps files.")
# else:
#     print("\nNo supplementary tasks were run, skipping processing.")

In [ ]:
# Process Data
if tasks_to_get:
    maps_supplement_raw_json_dir = os.path.join(temp_dir, "maps_supplement")

    # log file to skip existing
    processed_log_path = os.path.join(maps_supplement_raw_json_dir, "processed_files.log")
    try:
        with open(processed_log_path, 'r') as f:
            processed_files_set = set(line.strip() for line in f)
        print(f"Loaded {len(processed_files_set)} records from the processed file log.")
    except FileNotFoundError:
        processed_files_set = set()
        print("No processed file log found. Will process all files.")

    supplement_json_files = glob.glob(os.path.join(maps_supplement_raw_json_dir, "*.json"))
    new_supplement_data = []
    files_processed_this_run = []

    print(f"\nFound {len(supplement_json_files)} total supplementary JSON files.")

    for file_path in supplement_json_files:
        file_name = os.path.basename(file_path)

        # skip if in log
        if file_name in processed_files_set:
            print(f"  Skipping already processed file: {file_name}")
            continue

        print(f"  Processing new supplementary file: {file_name}")

        # process resuly
        parsed_data = parse_maps_results(file_path)
        if parsed_data:
            new_supplement_data.extend(parsed_data)
            files_processed_this_run.append(file_name)
        else:
            print(f"  No data extracted from {file_name}.")

    # merge , deduplicate
    if new_supplement_data:
        df_new = pd.DataFrame(new_supplement_data)
        print(f"\nProcessed {len(df_new)} new records from {len(files_processed_this_run)} new files.")
        supplement_output_dir = os.path.join(output_dir, "supplement")
        os.makedirs(supplement_output_dir, exist_ok=True)
        supplement_csv_path = os.path.join(supplement_output_dir, "supplement_processed_summary.csv")

        try:
            df_old = pd.read_csv(supplement_csv_path)
            print(f"Loaded {len(df_old)} existing records from CSV.")
            df_all = pd.concat([df_old, df_new], ignore_index=True)
        except (FileNotFoundError, pd.errors.EmptyDataError):
            print("No existing CSV found. Using new data only.")
            df_all = df_new

        # deduplicate, sort
        print(f"Total records before deduplication: {len(df_all)}")
        supplement_deduplicated_df = df_all.drop_duplicates(subset=['title', 'address'], keep='first')
        supplement_sorted_df = supplement_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')
        print(f"After deduplication, {len(supplement_sorted_df)} unique records remain.")

        # overlap by clean data
        supplement_sorted_df.to_csv(supplement_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSaved final processed supplementary data to: '{supplement_csv_path}'")
        display(supplement_sorted_df.head())

        # update log
        with open(processed_log_path, 'a') as f:
            for file_name in files_processed_this_run:
                f.write(f"{file_name}\n")
        print(f"Updated 'processed_files.log' with {len(files_processed_this_run)} new filenames.")

    else:
        print("\nNo new supplementary files to process.")
else:
    print("\nNo supplementary tasks were run, skipping processing.")

# Merge and process

In [ ]:
print("--- Final Merging and Consolidation ---")

output_dir = os.path.join(base_path, "output")

original_merged_path = os.path.join(output_dir, "merged_deduplicated", "merged_deduplicated_summary.csv")
supplement_path = os.path.join(output_dir, "supplement", "supplement_processed_summary.csv")

try:
    original_df = pd.read_csv(original_merged_path)
    print(f"Loaded original merged data: {original_df.shape}")
except FileNotFoundError:
    print(f"Error: '{original_merged_path}' not found.")
    original_df = pd.DataFrame()

try:
    supplement_df = pd.read_csv(supplement_path)
    print(f"Loaded supplementary data: {supplement_df.shape}")
except FileNotFoundError:
    print(f"'{supplement_path}' not found.")
    supplement_df = pd.DataFrame()


if not original_df.empty:
    if not supplement_df.empty:
        print("\\nPreparing supplementary data...")
        supplement_df['isFinder'] = 0
        supplement_df['isMap'] = 1
        supplement_df['isBoth'] = 0

        supplement_df = supplement_df.reindex(columns=original_df.columns)

        # Concatenate
        combined_df = pd.concat([original_df, supplement_df], ignore_index=True)
        print(f"\\nShape before final consolidation: {combined_df.shape}")

        agg_funcs = {
            # For all columns except 'title', take the first non-null value within each group
            col: 'first' for col in combined_df.columns if col != 'title'
        }
        # If any row has a 1, the result is 1
        agg_funcs['isFinder'] = 'max'
        agg_funcs['isMap'] = 'max'

        if 'keywords' in agg_funcs:
            agg_funcs['keywords'] = 'first'
        if 'category' in agg_funcs:
            agg_funcs['category'] = 'first'

        final_df = combined_df.groupby('title', as_index=False).agg(agg_funcs)

        # Recalculate the 'isBoth' flag for accuracy based on the final merged flags
        final_df['isBoth'] = ((final_df['isFinder'] == 1) & (final_df['isMap'] == 1)).astype(int)

        print(f"Shape after consolidation: {final_df.shape}")

    else:
        # If there is no supplementary data, the final result is the original data
        final_df = original_df
        print("\\nNo supplementary data to merge.")

    # Sort and save the final result
    final_sorted_df = final_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

    final_output_dir = os.path.join(output_dir, "final")
    os.makedirs(final_output_dir, exist_ok=True)

    final_csv_path = os.path.join(final_output_dir, "final_processed.csv")
    final_sorted_df.to_csv(final_csv_path, index=False, encoding='utf-8-sig')

    print(f"\\nSaved final processed data to: '{final_csv_path}'")
    print("\\nFinal Processed Data Preview:")
    display(final_sorted_df.head(20))

else:
    print("Original merged data not found. Cannot create final processed file.")

# Rate Distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords/output/final/final_processed.csv"


try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

if not df.empty:
    # ZIP codes for each region
    blacksburg_zips = [24060, 24061, 24062, 24063]
    christiansburg_zips = [24068, 24073]
    roanoke_lynchburg_zips = [
        # Roanoke ZIPs
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        # Lynchburg ZIPs
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
    dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098,
        # NOVA) ZIP
        22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308, 22309, 22310,
        22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205, 22206, 22207,
        22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044, 22046, 22003,
        22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116, 22118, 22180,
        22181, 22182, 22067, 22039
    ]
    ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
    charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
    ]

    # map ZIP code to location
    def map_zip_to_location(zip_code):
        if pd.isna(zip_code): return 'Unknown'
        try:
            zip_code = int(zip_code)
        except ValueError:
            return 'Unknown'

        if zip_code in blacksburg_zips:
            return 'Blacksburg'
        elif zip_code in christiansburg_zips:
            return 'Christiansburg'
        elif zip_code in roanoke_lynchburg_zips:
            return 'Roanoke/Lynchburg'
        elif zip_code in dc_zips:
            return 'DC'
        elif zip_code in ny_zips:
            return 'NY'
        elif zip_code in charlotte_zips:
            return 'Charlotte'
        return 'Unknown'

    df['location_name'] = df['zip'].apply(map_zip_to_location)

    df_filtered = df[(df['category'] != 'Unknown') & (df['location_name'] != 'Unknown')].copy()
    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)
    # low rate dummy
    df_filtered['low_rating_dummy'] = np.where(df_filtered['rating_value'] <= 3, 1, 0)
    urban_locations = ['DC', 'NY', 'Charlotte']
    df_filtered['location_type'] = df_filtered['location_name'].apply(lambda x: 'Urban' if x in urban_locations else 'Country')
    # std rate 1-5
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0)
    df_filtered['rating_std'] = df_filtered[star_cols].std(axis=1)

    print(f"After all filtering and preparation, {len(df_filtered)} records remain for analysis.")


    # Boxplot
    categories = sorted(df_filtered['category'].unique())
    locations_order = sorted(df_filtered['location_name'].unique())

    print(f"---Generating boxplots for {len(categories)} categories---")

    for category in categories:
        plt.figure(figsize=(12, 7))
        category_df = df_filtered[df_filtered['category'] == category]

        sns.boxplot(x='location_name', y='rating_value', data=category_df, order=locations_order)

        plt.title(f'Rating Distribution for: {category}', fontsize=16)
        plt.xlabel('Region', fontsize=12)
        plt.ylabel('Rating Value', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # Histograms each region
    print(f"---Generating Histograms for {len(categories)} categories---")
    for category in categories:
        for location in locations_order:
            plt.figure(figsize=(10, 6))

            plot_df = df_filtered[
                (df_filtered['category'] == category) &
                (df_filtered['location_name'] == location)
            ]

            if plot_df.empty:
                print(f"  Skipping histogram for '{category}' in '{location}' (No data)")
                plt.close()
                continue

            sns.histplot(data=plot_df, x='rating_value', bins=15, kde=True, color='blue')

            plt.title(f'Histogram of Ratings for: {category} in {location}')
            plt.xlabel('Rating Value')
            plt.ylabel('Count of Clinics')
            plt.xlim(1, 5)
            plt.tight_layout()
            plt.show()

    # Violin Plot of Rating Value
    print(f"---Generating Violin Plot for {len(categories)} categories---")
    for category in categories:
        plt.figure(figsize=(12, 7))
        category_df = df_filtered[df_filtered['category'] == category]
        sns.violinplot(x='location_name', y='rating_value', data=category_df, order=locations_order)
        plt.title(f'Rating Distribution for: {category}', fontsize=16)
        plt.xlabel('Region', fontsize=12)
        plt.ylabel('Rating Value', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # low rate crosstab
    low_rating_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_dummy'])
    print("--- Low Rating (<=3) Analysis: Urban vs. Rural ---")
    display(low_rating_crosstab)

    low_rating_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title('Count of Low Ratings (<=3) by Location Type')
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.xticks(rotation=0)
    plt.legend(title='Low Rating (<=3)', labels=['No (>3)', 'Yes (<=3)'])
    plt.tight_layout()
    plt.show()

    #std
    std_by_category_location_series = df_filtered.groupby(['category', 'location_name'])['rating_std'].mean()
    print(std_by_category_location_series.unstack())
    std_by_category_location = df_filtered.groupby(['category', 'location_name'], as_index=False)['rating_std'].mean()
    print("--- Average Rating Standard Deviation by Category and Location ---")


    if not std_by_category_location.empty:
        for category in categories:
            plt.figure(figsize=(12, 7))

            plot_df = std_by_category_location[std_by_category_location['category'] == category]

            if plot_df.empty:
                print(f"  Skipping std dev plot for '{category}' (No data)")
                plt.close()
                continue

            # Applied fix
            sns.barplot(x='location_name', y='rating_std', data=plot_df, order=locations_order, palette="coolwarm", hue='location_name', legend=False)

            plt.title(f'Average Rating Standard Deviation for: {category}', fontsize=16)
            plt.xlabel('Region', fontsize=12)
            plt.ylabel('Average Standard Deviation of Star Counts', fontsize=12)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.savefig(f'std_dev_ratings_cat_{category}.png')
            plt.show()
    else:
        print("Could not generate standard deviation data from groupby.")

# NPI records as Input

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

base_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords"
nppes_data_path = "/content/drive/MyDrive/RA/health_care/NPI"
main_npi_file = "npidata_pfile_20050523-20251012.csv"

# ZIP codes for each region
blacksburg_zips = [24060, 24061, 24062, 24063]
christiansburg_zips = [24068, 24073]
roanoke_lynchburg_zips = [
        # Roanoke ZIPs
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        # Lynchburg ZIPs
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098,
        # NOVA ZIPs
        22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308, 22309, 22310,
        22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205, 22206, 22207,
        22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044, 22046, 22003,
        22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116, 22118, 22180,
        22181, 22182, 22067, 22039
    ]
ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
]

all_target_zips = blacksburg_zips + christiansburg_zips + roanoke_lynchburg_zips + dc_zips + charlotte_zips + ny_zips

exclusion_taxonomy_codes = [
    # === Suppliers (DME, Pharmacy, Labs) ===
    '332B00000X',  # Durable Medical Equipment & Medical Supplies
    '332S00000X',  # Hearing Aid Equipment
    '333600000X',  # Pharmacy (General)
    '3336C0003X',  # Community/Retail Pharmacy
    '335E00000X',  # Prosthetic/Orthotic Supplier
    '335U00000X',  # Ocularist
    '291U00000X',  # Clinical Medical Laboratory

    # === Transportation Services ===
    '341600000X',  # Ambulance
    '343900000X',  # Non-emergency Medical Transport (VAN)
    '347B00000X',  # Taxi
    '347C00000X',  # Air Carrier

    # === Managed Care / Insurance ===
    '302F00000X',  # Health Maintenance Organization (HMO)
    '305S00000X',  # Preferred Provider Organization (PPO)

    # === Government Agencies ===
    '251K00000X',  # Public Health or Welfare Agency
    '251S00000X',  # Community Health
    '251T00000X',  # Military/U.S. Coast Guard Transport
    '251V00000X',  # Local Education Agency (LEA)

    # === Other Service & Social Work Providers ===
    '174400000X',  # Social Worker
    '1041C0700X',  # Clinical Social Worker
    '171M00000X',  # Case Manager/Care Coordinator

    # === Schools & Students ===
    '352Y00000X',  # Local Education Agency (as an organization)
    '390200000X',  # Student in an Organized Health Care Education/Training Program

    # === Residential & Long-Term Care Facilities ===
    '314000000X',  # Skilled Nursing Facility (SNF)
    '310400000X',  # Assisted Living Facility
    '320800000X',  # Residential Treatment Facility for Children
    '324500000X',  # Substance Abuse Rehabilitation Facility

    # === Community & Home Health Agencies ===
    '251E00000X',  # Home Health Agency
    '251C00000X',  # Community/Behavioral Health Agency
    '251J00000X',  # Voluntary or Charitable Agency
    '251F00000X'   # Hospice
]

# Taxonomy Code Distribution
print("--- Analyzing Taxonomy Code Distribution ---")
try:
    taxonomy_iterator = pd.read_csv(
        os.path.join(nppes_data_path, main_npi_file),
        usecols=['Healthcare Provider Taxonomy Code_1'],
        chunksize=100000,
        low_memory=False,
        encoding='utf-8'
    )
    taxonomy_counts = pd.Series(dtype='int64')
    for chunk in taxonomy_iterator:
        # Filter out excluded taxonomy codes before counting
        filtered_chunk = chunk[~chunk['Healthcare Provider Taxonomy Code_1'].isin(exclusion_taxonomy_codes)].copy()
        taxonomy_counts = taxonomy_counts.add(filtered_chunk['Healthcare Provider Taxonomy Code_1'].value_counts(), fill_value=0)

    taxonomy_counts = taxonomy_counts.sort_values(ascending=False)
    total_providers = taxonomy_counts.sum()
    taxonomy_percentage = (taxonomy_counts / total_providers) * 100

    print("\nTop 10 Healthcare Provider Taxonomy Categories:")
    top_10_taxonomy = taxonomy_percentage.head(10)
    print(top_10_taxonomy.to_string())

    plt.figure(figsize=(12, 8))
    sns.barplot(x=top_10_taxonomy.values, y=top_10_taxonomy.index, palette="viridis")
    plt.title('Top 10 Healthcare Provider Taxonomy Code Distribution', fontsize=16)
    plt.xlabel('Percentage (%)', fontsize=12)
    plt.ylabel('Taxonomy Code', fontsize=12)
    plt.tight_layout()
    plt.show()

except FileNotFoundError as e:
    print(f"\nError: Could not find the main NPI file for taxonomy analysis: {e}")

print("--- Processing Main NPI Data ---")
main_columns_to_load = [
    'NPI', 'Entity Type Code',
    'Provider Organization Name (Legal Business Name)',
    'Provider Last Name (Legal Name)', 'Provider First Name',
    'Provider Credential Text', 'Provider Enumeration Date',
    'Provider Business Practice Location Address Postal Code'
] + [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]

all_regional_data_chunks = []
print("Processing main NPI file in chunks to find all providers in target ZIPs...")

try:
    chunk_iterator = pd.read_csv(
        os.path.join(nppes_data_path, main_npi_file),
        usecols=main_columns_to_load,
        chunksize=100000,
        low_memory=False,
        encoding='utf-8'
    )

    for i, chunk in enumerate(chunk_iterator):
        print(f"  - Processing chunk {i+1}...")

        taxonomy_cols = [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]
        exclusion_mask = chunk[taxonomy_cols].isin(exclusion_taxonomy_codes).any(axis=1)
        filtered_chunk = chunk[~exclusion_mask].copy()

        if not filtered_chunk.empty:
            filtered_chunk['zip'] = filtered_chunk['Provider Business Practice Location Address Postal Code'].astype(str).str.split('-').str[0]
            filtered_chunk.dropna(subset=['zip'], inplace=True)
            filtered_chunk['zip'] = pd.to_numeric(filtered_chunk['zip'], errors='coerce')
            filtered_chunk.dropna(subset=['zip'], inplace=True)
            filtered_chunk['zip'] = filtered_chunk['zip'].astype(int)

            regional_providers = filtered_chunk[filtered_chunk['zip'].isin(all_target_zips)]

            if not regional_providers.empty:
                all_regional_data_chunks.append(regional_providers)

    if all_regional_data_chunks:
        df_regional = pd.concat(all_regional_data_chunks, ignore_index=True)


        df_regional['Name'] = df_regional['Provider Organization Name (Legal Business Name)'].copy()
        individual_mask = df_regional['Entity Type Code'] == 1
        df_regional.loc[individual_mask, 'Name'] = df_regional.loc[individual_mask, 'Provider First Name'] + ' ' + df_regional.loc[individual_mask, 'Provider Last Name (Legal Name)']

        # Rename columns
        df_regional.rename(columns={
            'Provider Enumeration Date': 'Registration_Date',
            'Provider Credential Text': 'Credential',
            'Healthcare Provider Taxonomy Code_1': 'Taxonomy_Code'
        }, inplace=True)

        def map_zip_to_location(zip_code):
            if zip_code in blacksburg_zips: return 'blacksburg'
            if zip_code in christiansburg_zips: return 'christiansburg'
            if zip_code in roanoke_lynchburg_zips: return 'roanoke_lynchburg'
            if zip_code in dc_zips: return 'washington_dc'
            if zip_code in charlotte_zips: return 'charlotte_nc'
            if zip_code in ny_zips: return 'new_york_ny'
            return 'Unknown'

        df_regional['location_name'] = df_regional['zip'].apply(map_zip_to_location)

        final_columns = [
            'NPI', 'Name', 'Registration_Date', 'Credential', 'zip', 'location_name', 'Taxonomy_Code'
        ]

        df_final_output = df_regional[final_columns].copy()
        df_final_output.dropna(subset=['Name'], inplace=True)
        df_final_output.drop_duplicates(subset=['NPI'], inplace=True)

        print(f"\nProcessing complete. Extracted {len(df_final_output)} unique providers from target regions.")

        print("\n--- Final Data Preview ---")
        display(df_final_output.head())

        # Optional: Save to a new CSV
        output_dir = os.path.join(base_path, "temp", "npi_processed")
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, "npi_processed.csv")
        df_final_output.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"\nData saved to {output_path}")

    else:
        print("No regional providers found.")
        df_final_output = pd.DataFrame()

except FileNotFoundError as e:
    print(f"\nError: Could not find the main NPI file: {e}")
    df_final_output = pd.DataFrame()

In [ ]:
# dentist
import pandas as pd
from IPython.display import display

dentist_taxonomy_codes = [
    '122300000X',  # Dentist
    '1223D0001X',  # Dental Public Health
    '1223E0200X',  # Endodontics
    '1223G0001X',  # General Practice
    '1223P0106X',  # Pediatric Dentistry
    '1223P0221X',  # Periodontics
    '1223P0300X',  # Prosthodontics
    '1223P0700X',  # Pediatric Dentistry (alternate code)
    '1223S0112X',  # Oral and Maxillofacial Surgery
    '1223X0400X',  # Orthodontics and Dentofacial Orthopedics
    '122400000X',  # Dental Assistant
    '124Q00000X',  # Dental Hygienist
    '126800000X',  # Dental Laboratory
    '1223X2210X',  # Orofacial Pain
    '1223X0008X',  # Oral and Maxillofacial Radiology
    '1223P0106X',  # Oral and Maxillofacial Pathology
    '1223D0004X'   # Dentist Anesthesiologist
]

TAXONOMY_CATEGORIES = {
    'General': ['122300000X', '1223G0001X', '1223D0001X', '122400000X', '124Q00000X', '126800000X'],
    'Specialist': ['1223E0200X', '1223P0106X', '1223P0221X', '1223P0300X', '1223P0700X', '1223X0400X','1223X2210X','1223X0008X'],
    'Surgery': ['1223S0112X']
}

def get_npi_category(taxonomy_code):
    for category, codes in TAXONOMY_CATEGORIES.items():
        if taxonomy_code in codes:
            return category
    return 'Unknown'


if 'df_final_output' in locals() and not df_final_output.empty:
    print("Processing NPI filtered data...")
    df_dentists_filtered = df_final_output[df_final_output['Taxonomy_Code'].isin(dentist_taxonomy_codes)].copy()
    print(f"Filtered to {len(df_dentists_filtered)} dental-related providers.")
    df_dentists_filtered['category'] = df_dentists_filtered['Taxonomy_Code'].apply(get_npi_category)
    df_npi_keywords = df_dentists_filtered[[
        'Name', 'location_name', 'category', 'Registration_Date', 'Credential'
    ]].copy()

    df_npi_keywords.rename(columns={'Name': 'keyword'}, inplace=True)

    print(f"\nCreated 'df_npi_keywords' DataFrame with {len(df_npi_keywords)} entries for the next step.")
    print("\n--- Preview of the data to be posted ---")
    display(df_npi_keywords.head())

else:
    print("Error: 'df_final_output' not found or is empty. Please run the previous cell successfully.")
    df_npi_keywords = pd.DataFrame()

In [ ]:
# Post NPI Keywords to DataForSEO Maps API
print("--- Starting Task POST for NPI Keywords ---")

if 'df_npi_keywords' in locals() and not df_npi_keywords.empty:
    npi_search_temp_dir = os.path.join(base_path, "temp", "npi_map_search")
    os.makedirs(npi_search_temp_dir, exist_ok=True)
    task_list_csv_path = os.path.join(npi_search_temp_dir, "task_list.csv")

    # Load existing tasks to skip
    existing_tasks = set()
    try:
        if os.path.exists(task_list_csv_path):
            tasks_df = pd.read_csv(task_list_csv_path)
            if not tasks_df.empty:
                for index, row in tasks_df.iterrows():
                    existing_tasks.add((row['location_name'], row['keyword']))
            print(f"Loaded {len(existing_tasks)} existing tasks to skip.")
    except pd.errors.EmptyDataError:
        print("Task list file is empty. Starting fresh.")

    locations_to_search = {
        'roanoke_lynchburg': 200573,
        'blacksburg': 1027041,
        'christiansburg': 1027077,
        'washington_dc': 2840,
        'charlotte_nc': 200517,
        'new_york_ny' : 21167
    }

    maps_api_url = "https://api.dataforseo.com/v3/serp/google/maps/task_post"
    write_header = not os.path.exists(task_list_csv_path) or (os.path.getsize(task_list_csv_path) == 0)

    # Group keywords by their location
    grouped_keywords = df_npi_keywords.groupby('location_name')

    with open(task_list_csv_path, 'a', newline='', encoding='utf-8-sig') as f:
        import csv
        writer = csv.writer(f)
        if write_header:
            writer.writerow(["task_id", "api_type", "location_name", "category", "keyword",
                             "Registration_Date", "Credential", "raw_json_path"])
        # Iterate through each location group
        for loc_name, group in grouped_keywords:
            if loc_name not in locations_to_search:
                print(f"\nSkipping location '{loc_name}' as it has no corresponding location code.")
                continue

            loc_code = locations_to_search[loc_name]
            print(f"\nProcessing tasks for location: '{loc_name}' using code {loc_code}")

            for index, row in group.iterrows():
                keyword = row['keyword']
                category = row['category']
                reg_date = row['Registration_Date']
                credential = row['Credential']

                # Skip task already exists
                if (loc_name, keyword) in existing_tasks:
                    print(f"  Skipping already posted task for keyword: '{keyword}'")
                    continue

                task_id = post_dataforseo_task(maps_api_url, loc_code, keyword)

                if task_id:
                    safe_keyword = "".join(c for c in keyword if c.isalnum() or c in (' ', '_')).rstrip().replace(' ', '_')
                    raw_json_filename = f"{loc_name}_{category}_{safe_keyword[:100]}_maps.json"
                    raw_json_path = os.path.join(npi_search_temp_dir, raw_json_filename)

                    writer.writerow([task_id, "maps", loc_name, category, keyword,
                                     reg_date, credential, raw_json_path])
                    f.flush()

                time.sleep(1)

    print(f"\nAll targeted NPI keyword tasks posted.")
    print("!!! IMPORTANT: Please wait 20 minutes before running the next step. !!!")
else:
    print("Keyword DataFrame ('df_npi_keywords') is empty")

In [ ]:
base_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords"
npi_search_temp_dir = os.path.join(base_path, "temp", "npi_map_search")
task_list_csv_path = os.path.join(npi_search_temp_dir, "task_list.csv")
final_output_dir = os.path.join(base_path, "output", "final")
final_csv_path = os.path.join(final_output_dir, "npi_map_search_final.csv")
os.makedirs(final_output_dir, exist_ok=True)

tasks_to_get = []
try:
    tasks_df = pd.read_csv(task_list_csv_path)
    tasks_to_get = tasks_df.to_dict('records')
    print(f"Found {len(tasks_to_get)} tasks in the task list.")

    for task in tasks_to_get:
        # Skip if the JSON file already exists
        if os.path.exists(task['raw_json_path']):
            print(f"  Skipping GET: Result file already exists for Keyword='{task['keyword']}'")
            continue

        print(f" Getting results for: Location='{task['location_name']}', Keyword='{task['keyword']}'")
        get_dataforseo_results(task['task_id'], task['api_type'], task['raw_json_path'])
        time.sleep(1)

    print("\nAll result retrieval attempts completed.")

except FileNotFoundError:
    print(f"Error: Task list file not found at '{task_list_csv_path}'. Cannot get results.")




In [ ]:
def parse_maps_results_with_category(file_path, category):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

    extracted_data = []
    if not (data and data.get("tasks") and data["tasks"][0].get("result")):
        return []

    for result in data["tasks"][0]["result"]:
        if not result or not result.get("items"):
            continue
        for item in result["items"]:
            rating = item.get("rating", {})
            if not isinstance(rating, dict): rating = {}
            address_info = item.get("address_info", {})
            if not isinstance(address_info, dict): address_info = {}
            rating_distribution = item.get("rating_distribution", {})
            if not isinstance(rating_distribution, dict): rating_distribution = {}

            record = {
                "title": item.get("title"),
                "address": item.get("address"),
                "latitude": item.get("latitude"),
                "longitude": item.get("longitude"),
                "zip": address_info.get("zip"),
                "rating_value": rating.get("value"),
                "votes_count": rating.get("votes_count"),
                "type": item.get("type"),
                "category": category
            }
            # Add star ratings
            for i in range(1, 6):
                record[f'rating_{i}_star'] = rating_distribution.get(str(i), 0)

            extracted_data.append(record)
    return extracted_data



In [ ]:
# if not tasks_df.empty:
#     npi_maps_data = []
#     print(f"\nProcessing {len(tasks_df)} task results...")

#     for index, task_row in tasks_df.iterrows():
#         file_path = task_row['raw_json_path']
#         category = task_row['category']

#         if os.path.exists(file_path):
#             npi_maps_data.extend(parse_maps_results_with_category(file_path, category))
#         else:
#             print(f"  Warning: JSON file not found for task '{task_row['keyword']}', skipping.")

#     if npi_maps_data:
#         df_processed = pd.DataFrame(npi_maps_data)
#         print(f"Parsed {len(df_processed)} total records from all JSON files.")

#         df_deduplicated = df_processed.drop_duplicates(subset=['title', 'address'], keep='first').copy()
#         print(f"After deduplication, {len(df_deduplicated)} unique records remain.")

#         df_sorted = df_deduplicated.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

#         final_columns = [
#             'title', 'category', 'address', 'latitude', 'longitude', 'zip',
#             'rating_value', 'votes_count', 'rating_1_star', 'rating_2_star',
#             'rating_3_star', 'rating_4_star', 'rating_5_star', 'type'
#         ]
#         for col in final_columns:
#             if col not in df_sorted.columns:
#                 df_sorted[col] = None

#         df_final_output = df_sorted[final_columns]

#         df_final_output.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
#         print(f"\nProcessing complete. Final data saved to: {final_csv_path}")

#         print("\nPreview of the final processed NPI Map Search data:")
#         display(df_final_output.head())
#     else:
#         print("\nNo data was extracted from the JSON result files.")
# else:
#     print("\nNo tasks were found in the task list, so no data to process.")

In [ ]:
def load_processed_log(log_path):
    # Load the set of filenames that have already been processed
    try:
        with open(log_path, 'r') as f:
            return set(line.strip() for line in f)
    except FileNotFoundError:
        return set()

def update_processed_log(log_path, new_files):
    # Append newly processed filenames to the log
    with open(log_path, 'a') as f:
        for file_name in new_files:
            f.write(f"{file_name}\n")

def load_existing_data(csv_path):
    # Load the previously processed CSV data
    try:
        return pd.read_csv(csv_path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

In [ ]:
if tasks_to_get:
    log_path = os.path.join(npi_search_temp_dir, "_processed_npi_files.log")

    # Load old data and log
    processed_files_set = load_processed_log(log_path)
    df_old = load_existing_data(final_csv_path)
    print(f"Loaded {len(processed_files_set)} processed file records from log.")
    print(f"Loaded {len(df_old)} existing records from '{os.path.basename(final_csv_path)}'.")

    # Process new files
    new_npi_maps_data = []
    files_processed_this_run = []

    print(f"\nProcessing {len(tasks_df)} task results...")
    for index, task_row in tasks_df.iterrows():
        file_path = task_row['raw_json_path']
        file_name = os.path.basename(file_path)
        category = task_row.get('category', 'Unknown')

        # Skip file already in log
        if file_name in processed_files_set:
            continue

        print(f"  Processing new file: {file_name}")

        if os.path.exists(file_path):
            parsed_data = parse_maps_results_with_category(file_path, category)
            if parsed_data:
                new_npi_maps_data.extend(parsed_data)
                files_processed_this_run.append(file_name)
        else:
            print(f"  Warning: JSON file not found for task '{task_row['keyword']}', skipping.")

    # Consolidate Data
    if new_npi_maps_data:
        df_new = pd.DataFrame(new_npi_maps_data)
        print(f"\nParsed {len(df_new)} new records from {len(files_processed_this_run)} new files.")

        df_all = pd.concat([df_new, df_old], ignore_index=True)
        print(f"Total records before deduplication: {len(df_all)}")

        # Deduplicate
        df_deduplicated = df_all.drop_duplicates(subset=['title', 'address'], keep='first').copy()
        print(f"After deduplication, {len(df_deduplicated)} unique records remain.")

        df_sorted = df_deduplicated.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

        final_columns = [
            'title', 'category', 'address', 'latitude', 'longitude', 'zip',
            'rating_value', 'votes_count', 'rating_1_star', 'rating_2_star',
            'rating_3_star', 'rating_4_star', 'rating_5_star', 'type'
        ]
        for col in final_columns:
            if col not in df_sorted.columns:
                df_sorted[col] = None

        df_final_output = df_sorted[final_columns]

        # Overwrite the final CSV with the clean data
        df_final_output.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nProcessing complete. Final data saved to: {final_csv_path}")

        print("\nPreview of the final processed NPI Map Search data:")
        display(df_final_output.head())

        # Update the log file with the new files
        update_processed_log(log_path, files_processed_this_run)
        print(f"Updated '{os.path.basename(log_path)}' with {len(files_processed_this_run)} new files.")
    else:
        print("\nNo new files to process. Data in CSV remains unchanged.")
else:
    print("\nNo tasks were found in the task list, so no data to process.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from IPython.display import display

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords/output/final/npi_map_search_final.csv"

try:
    df = pd.read_csv(final_csv_path)
    print(f"Successfully loaded '{os.path.basename(final_csv_path)}'")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

if not df.empty:
    blacksburg_zips = [24060, 24061, 24062, 24063]
    christiansburg_zips = [24068, 24073]
    roanoke_lynchburg_zips = [
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
    dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098, 22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308,
        22309, 22310, 22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205,
        22206, 22207, 22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044,
        22046, 22003, 22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116,
        22118, 22180, 22181, 22182, 22067, 22039
    ]
    ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
    charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
    ]

    def map_zip_to_location(zip_code):
        if pd.isna(zip_code): return 'Unknown'
        try:
            zip_code = int(zip_code)
        except ValueError:
            return 'Unknown'
        if zip_code in blacksburg_zips: return 'Blacksburg'
        if zip_code in christiansburg_zips: return 'Christiansburg'
        if zip_code in roanoke_lynchburg_zips: return 'Roanoke/Lynchburg'
        if zip_code in dc_zips: return 'DC'
        if zip_code in ny_zips: return 'NY'
        if zip_code in charlotte_zips: return 'Charlotte'
        return 'Unknown'

    df['location_name'] = df['zip'].apply(map_zip_to_location)

    df_filtered = df[(df['category'] != 'Unknown') & (df['location_name'] != 'Unknown')].copy()
    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)
    df_filtered['low_rating_dummy'] = np.where(df_filtered['rating_value'] <= 3, 1, 0)
    urban_locations = ['DC', 'NY', 'Charlotte']
    df_filtered['location_type'] = df_filtered['location_name'].apply(lambda x: 'Urban' if x in urban_locations else 'Rural') # Corrected 'Country' to 'Rural' for consistency
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0)
    df_filtered['rating_std'] = df_filtered[star_cols].std(axis=1)

    print(f"After all filtering and preparation, {len(df_filtered)} records remain for analysis.")


    # Boxplot
    categories = sorted(df_filtered['category'].unique())
    locations_order = sorted(df_filtered['location_name'].unique())

    print(f"---Generating boxplots for {len(categories)} categories---")

    for category in categories:
        plt.figure(figsize=(12, 7))
        category_df = df_filtered[df_filtered['category'] == category]

        sns.boxplot(x='location_name', y='rating_value', data=category_df, order=locations_order)

        plt.title(f'Rating Distribution for: {category}', fontsize=16)
        plt.xlabel('Region', fontsize=12)
        plt.ylabel('Rating Value', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # Histograms each region
    print(f"---Generating Histograms for {len(categories)} categories---")
    for category in categories:
        for location in locations_order:
            plt.figure(figsize=(10, 6))

            plot_df = df_filtered[
                (df_filtered['category'] == category) &
                (df_filtered['location_name'] == location)
            ]

            if plot_df.empty:
                print(f"  Skipping histogram for '{category}' in '{location}' (No data)")
                plt.close()
                continue

            sns.histplot(data=plot_df, x='rating_value', bins=15, kde=True, color='blue')

            plt.title(f'Histogram of Ratings for: {category} in {location}')
            plt.xlabel('Rating Value')
            plt.ylabel('Count of Clinics')
            plt.xlim(1, 5)
            plt.tight_layout()
            plt.show()

    # Violin Plot of Rating Value
    print(f"---Generating Violin Plot for {len(categories)} categories---")
    for category in categories:
        plt.figure(figsize=(12, 7))
        category_df = df_filtered[df_filtered['category'] == category]
        sns.violinplot(x='location_name', y='rating_value', data=category_df, order=locations_order)
        plt.title(f'Rating Distribution for: {category}', fontsize=16)
        plt.xlabel('Region', fontsize=12)
        plt.ylabel('Rating Value', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # low rate crosstab
    low_rating_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_dummy'])
    print("--- Low Rating (<=3) Analysis: Urban vs. Rural ---")
    display(low_rating_crosstab)

    low_rating_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title('Count of Low Ratings (<=3) by Location Type')
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.xticks(rotation=0)
    plt.legend(title='Low Rating (<=3)', labels=['No (>3)', 'Yes (<=3)'])
    plt.tight_layout()
    plt.show()

    #std
    std_by_category_location_series = df_filtered.groupby(['category', 'location_name'])['rating_std'].mean()
    print(std_by_category_location_series.unstack())
    std_by_category_location = df_filtered.groupby(['category', 'location_name'], as_index=False)['rating_std'].mean()
    print("--- Average Rating Standard Deviation by Category and Location ---")


    if not std_by_category_location.empty:
        for category in categories:
            plt.figure(figsize=(12, 7))

            plot_df = std_by_category_location[std_by_category_location['category'] == category]

            if plot_df.empty:
                print(f"  Skipping std dev plot for '{category}' (No data)")
                plt.close()
                continue

            # Applied fix
            sns.barplot(x='location_name', y='rating_std', data=plot_df, order=locations_order, palette="coolwarm", hue='location_name', legend=False)

            plt.title(f'Average Rating Standard Deviation for: {category}', fontsize=16)
            plt.xlabel('Region', fontsize=12)
            plt.ylabel('Average Standard Deviation of Star Counts', fontsize=12)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.savefig(f'std_dev_ratings_cat_{category}.png')
            plt.show()
    else:
        print("Could not generate standard deviation data from groupby.")